# 🧭 Pathfinding Lab
### Interactive Algorithm Visualiser

This notebook embeds a fully interactive pathfinding visualiser.

**Algorithms included:**
- **A\*** — Optimal, heuristic-guided search
- **Dijkstra** — Optimal, weighted shortest path
- **BFS** — Optimal for unweighted graphs
- **Greedy Best-First** — Fast but non-optimal
- **DFS** — Explores deep, non-optimal
- **Bidirectional BFS** — Searches from both ends

**How to use:**
1. Run the cell below to launch the visualiser
2. Select a **Tool** from the left panel (Wall, Start, End, Moving Obstacle, Erase)
3. Click/drag on the grid to place elements
4. Choose an **Algorithm**
5. Press **▶ Visualise** to animate the search
6. Use **Step →** to walk through one node at a time
7. Try **Random Maze** for a quick demo!

> 💡 Moving obstacles can be dragged around with the Moving Obstacle tool selected.

In [ ]:
# ── Pathfinding Lab Launcher ──────────────────────────────────────────
# Run this cell to launch the interactive visualiser inside Colab.
# The HTML file is embedded directly — no external server needed.

from IPython.display import HTML, display
import os

# Option A: Load from uploaded file (upload pathfinding_visualiser.html first)
HTML_FILE = 'pathfinding_visualiser.html'

if os.path.exists(HTML_FILE):
    with open(HTML_FILE, 'r', encoding='utf-8') as f:
        html_content = f.read()
    display(HTML(f'<div style="width:100%;">{html_content}</div>'))
else:
    print(f"File '{HTML_FILE}' not found.")
    print("Please upload pathfinding_visualiser.html using the Files panel on the left,")
    print("then re-run this cell.")


---
## 📖 Algorithm Notes

| Algorithm | Optimal? | Complete? | Time Complexity | Notes |
|---|---|---|---|---|
| **A\*** | ✅ Yes | ✅ Yes | O(E log V) | Best general-purpose; uses Manhattan heuristic |
| **Dijkstra** | ✅ Yes | ✅ Yes | O(E log V) | Like A\* without heuristic; great for weighted graphs |
| **BFS** | ✅ Yes (unweighted) | ✅ Yes | O(V + E) | Guarantees shortest path in unweighted grids |
| **Greedy BFS** | ❌ No | ✅ Yes | O(E log V) | Fast but can miss shortest path |
| **DFS** | ❌ No | ✅ Yes | O(V + E) | Goes deep first; path often non-optimal |
| **Bidirectional BFS** | ✅ Yes (unweighted) | ✅ Yes | O(b^(d/2)) | Much faster than BFS on large grids |

> **V** = vertices (cells), **E** = edges (connections), **b** = branching factor, **d** = depth of solution

In [ ]:
# ── Export / Save standalone HTML ────────────────────────────────────
# Run this cell to download the standalone HTML file to your machine.
# The downloaded file works in any browser — no Colab needed.

from google.colab import files

if os.path.exists(HTML_FILE):
    files.download(HTML_FILE)
    print(f"✅ Downloading '{HTML_FILE}'...")
else:
    print(f"❌ '{HTML_FILE}' not found. Make sure it's uploaded first.")


---
## 🐍 Bonus: Pure Python Algorithm Demo

Run the cells below to see each algorithm implemented in plain Python — useful for understanding the logic without the visual layer.

In [ ]:
# ── Python reference implementations ─────────────────────────────────
import heapq
from collections import deque

# Simple 10x10 grid: 0=empty, 1=wall
DEMO_GRID = [
    [0,0,0,0,1,0,0,0,0,0],
    [0,1,1,0,1,0,1,1,1,0],
    [0,1,0,0,0,0,0,0,1,0],
    [0,1,0,1,1,1,1,0,1,0],
    [0,0,0,0,0,0,1,0,0,0],
    [1,1,1,1,0,1,1,0,1,1],
    [0,0,0,1,0,0,0,0,1,0],
    [0,1,0,1,1,1,1,0,0,0],
    [0,1,0,0,0,0,1,1,1,0],
    [0,0,0,0,0,0,0,0,0,0],
]
ROWS, COLS = 10, 10
START, END = (0, 0), (9, 9)

def neighbors(grid, r, c):
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] == 0:
            yield nr, nc

def reconstruct_path(came_from, end):
    path, cur = [], end
    while cur in came_from:
        path.append(cur)
        cur = came_from[cur]
    path.append(cur)
    return path[::-1]

def bfs(grid, start, end):
    queue = deque([start])
    came_from = {}
    visited = {start}
    while queue:
        cur = queue.popleft()
        if cur == end:
            return reconstruct_path(came_from, end)
        for nb in neighbors(grid, *cur):
            if nb not in visited:
                visited.add(nb)
                came_from[nb] = cur
                queue.append(nb)
    return []

def dijkstra(grid, start, end):
    pq = [(0, start)]
    dist = {start: 0}
    came_from = {}
    while pq:
        d, cur = heapq.heappop(pq)
        if cur == end:
            return reconstruct_path(came_from, end)
        for nb in neighbors(grid, *cur):
            nd = d + 1
            if nd < dist.get(nb, float('inf')):
                dist[nb] = nd
                came_from[nb] = cur
                heapq.heappush(pq, (nd, nb))
    return []

def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar(grid, start, end):
    pq = [(manhattan(start, end), 0, start)]
    g_score = {start: 0}
    came_from = {}
    while pq:
        _, g, cur = heapq.heappop(pq)
        if cur == end:
            return reconstruct_path(came_from, end)
        for nb in neighbors(grid, *cur):
            ng = g + 1
            if ng < g_score.get(nb, float('inf')):
                g_score[nb] = ng
                came_from[nb] = cur
                heapq.heappush(pq, (ng + manhattan(nb, end), ng, nb))
    return []

# ── Run & compare ──
import time

for name, fn in [('BFS', bfs), ('Dijkstra', dijkstra), ('A*', astar)]:
    t0 = time.perf_counter()
    path = fn(DEMO_GRID, START, END)
    ms = (time.perf_counter() - t0) * 1000
    print(f"{name:12s} → path length: {len(path):3d} | time: {ms:.4f} ms")
